# Foraging Cache — DuckDB Query Examples

The production parquet database lives on S3:
- `s3://aind-scratch-data/aind-dynamic-foraging-cache/session_table.parquet` — flat session table (one row per session)
- `.../trial_table/` — Hive-partitioned by `subject_id` (one row per trial)
- `.../event_table/` — Hive-partitioned by `subject_id` (one row per behavioral event)

DuckDB reads `s3://` natively and the cache bucket is **public** — no AWS credentials or setup are required. The paths (`SESSION_DB`, `TRIAL_DB`, `EVENT_DB`) are importable from `aind_dynamic_foraging_data_utils.foraging_cache`; reassign them to a local directory to query a local build instead.

**Primary pattern**: filter the session table, then JOIN to the trial/event tables so every row carries the session-level metadata (subject, date, task, `foraging_eff`, …) alongside the trial data.

Three options when reading the partitioned tables:
- `hive_partitioning=true` — partition-level pruning on `subject_id`
- `union_by_name=true` — merges column schemas across files (different NWB readers produce different column sets; missing columns fill with NULL)
- `CAST(subject_id AS VARCHAR)` — DuckDB infers the `subject_id=<n>` directory name as BIGINT; cast to VARCHAR to compare against the session table

> 💡 **New to DuckDB?** Paste this module's [`README.md`](README.md) into your favorite LLM (Claude / ChatGPT / Cursor / …) as context and ask your question in plain English — it returns runnable DuckDB that follows the cache's conventions (the three read options, the `subject_id` cast, and `subject_id` / `session_date` / `session_id` as leading keys). The cells below are hand-written examples of the same patterns. **Using a coding agent** (Claude Code, Codex, OpenCode)? Load the `aind-dynamic-foraging-data-access` skill from this repo's `.claude/skills/` instead of pasting the README.

In [1]:
import time

import duckdb

from aind_dynamic_foraging_data_utils.foraging_cache import SESSION_DB, TRIAL_DB, EVENT_DB

# SESSION_DB / TRIAL_DB / EVENT_DB point at the public production cache on S3.
# Reassign them to a local directory to query a local build instead.

# Reusable snippets - union_by_name handles schema differences across NWB reader paths.
READ_TRIALS = f"read_parquet('{TRIAL_DB}/**/*.parquet', hive_partitioning=true, union_by_name=true)"
READ_EVENTS = f"read_parquet('{EVENT_DB}/**/*.parquet', hive_partitioning=true, union_by_name=true)"

## 1. Quick start — the query helpers (start here)

Reach for the helpers first: `select_sessions` → `fetch_trials` / `fetch_events` cover the common
*"filter sessions, then fetch their trials/events"* workflow in two lines — session metadata is
joined on for you, and only the selected subjects' partition files are read, so it's fast. The
numbered sections below show the underlying **native DuckDB SQL**; use them when you need full
control (custom aggregations, window functions, trial↔event joins).

In [18]:
# Helpers — import alongside the paths from the setup cell above.
from aind_dynamic_foraging_data_utils.foraging_cache import (
    select_sessions, fetch_trials, fetch_events, read_trials,
)

# 1) Pick sessions: any predicate on the (small) session table — metric / metadata,
#    or subject-first via subjects=[...]. Extra `columns` are carried onto the trials.
sel = select_sessions(
    "task LIKE '%Uncoupled%' AND finished_trials > 500 AND finished_rate > 0.9",
    columns=["task", "finished_trials", "finished_rate"],
)
print(f"{len(sel):,} sessions over {sel['subject_id'].nunique()} subjects")

# 2) Fetch their trials — session metadata is joined onto every row.
trials = fetch_trials(sel, columns=["animal_response", "earned_reward",
                                    "reward_probabilityL", "reward_probabilityR"])
print(f"{len(trials):,} trials")
trials.head()

699 sessions over 290 subjects
406,349 trials


,subject_id,session_date,session_id,task,foraging_eff,finished_trials,animal_response,earned_reward,reward_probabilityL,reward_probabilityR
0,642860,2023-01-30,642860_2023-01-30_104448,Uncoupled Without Baiting,0.80867,666.0,1.0,True,0.9,0.9
1,642860,2023-01-30,642860_2023-01-30_104448,Uncoupled Without Baiting,0.80867,666.0,1.0,True,0.9,0.9
2,642860,2023-01-30,642860_2023-01-30_104448,Uncoupled Without Baiting,0.80867,666.0,1.0,True,0.9,0.9
3,642860,2023-01-30,642860_2023-01-30_104448,Uncoupled Without Baiting,0.80867,666.0,1.0,True,0.9,0.9
4,642860,2023-01-30,642860_2023-01-30_104448,Uncoupled Without Baiting,0.80867,666.0,1.0,True,0.9,0.9


In [3]:
# Events for the same sessions (optionally restricted to certain event types).
licks = fetch_events(sel, events=["left_lick_time", "right_lick_time"])
print(f"{len(licks):,} lick events")

# Need more than the helpers cover? read_trials(subjects) returns a fast, partition-scoped
# read_parquet(...) clause you drop into ANY SQL — here, a per-subject aggregate:
src = read_trials(sel["subject_id"].unique().tolist())
duckdb.sql(f"""
    SELECT subject_id, COUNT(*) AS n_trials, AVG(earned_reward::DOUBLE) AS reward_rate
    FROM {src} GROUP BY subject_id ORDER BY subject_id
""").df()

3,643,746 lick events


,subject_id,n_trials,reward_rate
0,642860,15892,0.336459
1,643987,5176,0.267002
2,647695,13117,0.346954
3,648585,14520,0.403857
4,648586,13704,0.419877
...,...,...,...
283,848325,2813,0.499467
284,848869,10514,0.458722
285,849570,8062,0.387001
286,850595,11403,0.400859


## 2. Database at a glance

A quick orientation to the whole cache. Session/mice counts come from one cheap scan of the flat session table; total trials and the timed load scan the partitioned trial table.

In [4]:
# Sessions, mice, and date span — one scan of the flat session table
duckdb.sql(f"""
    SELECT
        COUNT(*)                   AS total_sessions,
        COUNT(DISTINCT subject_id) AS total_mice,
        MIN(session_date)          AS first_session,
        MAX(session_date)          AS last_session
    FROM read_parquet('{SESSION_DB}')
""").df()

,total_sessions,total_mice,first_session,last_session
0,23867,902,2019-06-25,2026-06-03


In [5]:
# Distribution of data sources (which NWB route produced each session)
duckdb.sql(f"""
    SELECT nwb_data_source,
           COUNT(*)                   AS n_sessions,
           COUNT(DISTINCT subject_id) AS n_mice
    FROM read_parquet('{SESSION_DB}')
    GROUP BY nwb_data_source
    ORDER BY n_sessions DESC
""").df()

,nwb_data_source,n_sessions,n_mice
0,co_asset,15806,657
1,bpod_s3,4297,157
2,bonsai_s3,3764,340


In [6]:
# Total trials across the entire database (scans the partitioned trial table)
duckdb.sql(f"SELECT COUNT(*) AS total_trials FROM {READ_TRIALS}").df()

,total_trials
0,12541020


### Load the behavioral columns (+ `subject_id` / `session_date` / `session_id` keys) for **every** trial, and time it

Projecting just the keys + behavioral columns (choice / reward / reward-probabilities) keeps this fast and light even over the full ~12.5M-trial database on S3 — the normal starting point for a population analysis. Every result keeps `subject_id` and `session_date` so rows stay identifiable.

In [7]:
t0 = time.time()
df_5col = duckdb.sql(f"""
    SELECT subject_id, session_date, session_id, animal_response, earned_reward,
           reward_probabilityL, reward_probabilityR
    FROM {READ_TRIALS}
""").df()
elapsed = time.time() - t0

print(f"Loaded {len(df_5col):,} trials x {df_5col.shape[1]} columns "
      f"from {df_5col['session_id'].nunique():,} sessions "
      f"in {elapsed:.1f}s ({df_5col.memory_usage(deep=True).sum() / 1e9:.2f} GB in memory)")
df_5col.head()

Loaded 12,541,020 trials x 7 columns from 23,584 sessions in 8.3s (2.07 GB in memory)


,subject_id,session_date,session_id,animal_response,earned_reward,reward_probabilityL,reward_probabilityR
0,447921,2019-09-11,447921_2019-09-11_135646,2.0,False,0.0,1.0
1,447921,2019-09-11,447921_2019-09-11_135646,2.0,False,0.0,1.0
2,447921,2019-09-11,447921_2019-09-11_135646,1.0,True,0.0,1.0
3,447921,2019-09-11,447921_2019-09-11_135646,1.0,True,0.0,1.0
4,447921,2019-09-11,447921_2019-09-11_135646,1.0,True,0.0,1.0


## 3. Browse the session table

In [14]:
duckdb.sql(f"""
    SELECT _session_id, subject_id, session_date, task, finished_trials, foraging_eff
    FROM read_parquet('{SESSION_DB}')
    ORDER BY session_date DESC
    LIMIT 10
""").df()

,_session_id,subject_id,session_date,task,finished_trials,foraging_eff
0,854145_2026-06-03_131806,854145,2026-06-03,Uncoupled Baiting,413.0,0.684090
1,844917_2026-06-03_125658,844917,2026-06-03,Uncoupled Baiting,304.0,0.574562
2,844634_2026-06-03_122723,844634,2026-06-03,Uncoupled Baiting,179.0,0.797416
3,845710_2026-06-03_122619,845710,2026-06-03,Uncoupled Baiting,482.0,0.695470
4,854147_2026-06-03_122510,854147,2026-06-03,Uncoupled Baiting,464.0,0.596816
5,853577_2026-06-03_103236,853577,2026-06-03,Uncoupled Without Baiting,347.0,0.731159
6,853578_2026-06-03_103217,853578,2026-06-03,Uncoupled Without Baiting,523.0,0.703180
7,844636_2026-06-03_103213,844636,2026-06-03,Uncoupled Baiting,469.0,0.645705
8,849570_2026-06-03_102934,849570,2026-06-03,Uncoupled Without Baiting,499.0,0.710744
9,844920_2026-06-03_102832,844920,2026-06-03,Uncoupled Baiting,523.0,0.780315


## 4. Primary use case — filter sessions -> trials with session keys merged

Filter the session table however you like, then JOIN to the trial table so every trial row already carries the session-level metadata.

In [15]:
# Example query: trials from "well-trained" sessions —
#   curriculum = "Uncoupled Baiting", finished_trials > 500, finished_rate > 0.9.
# Filter the session table, then JOIN to the trial table (keys carried through).
df_trials = duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, finished_trials, finished_rate
        FROM read_parquet('{SESSION_DB}')
        WHERE curriculum_name = 'Uncoupled Baiting'
          AND finished_trials > 500
          AND finished_rate > 0.9
    )
    SELECT
        s.subject_id,
        s.session_date,
        t.session_id,
        t.trial,
        t.animal_response,
        t.earned_reward,
        t.reward_probabilityL,
        t.reward_probabilityR,
        t.rewarded_historyL,
        t.rewarded_historyR
    FROM {READ_TRIALS} t
    JOIN sel s ON t.session_id = s._session_id
    WHERE CAST(t.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    ORDER BY s.subject_id, s.session_date
""").df()

print(f"{len(df_trials):,} trials from {df_trials['session_id'].nunique():,} sessions")
df_trials.head(10)

2,670,497 trials from 4,672 sessions


,subject_id,session_date,session_id,trial,animal_response,earned_reward,reward_probabilityL,reward_probabilityR,rewarded_historyL,rewarded_historyR
0,703548,2024-02-23,703548_2024-02-23_84353,0.0,1.0,False,0.1,0.4,False,False
1,703548,2024-02-23,703548_2024-02-23_84353,1.0,1.0,True,0.1,0.4,False,True
2,703548,2024-02-23,703548_2024-02-23_84353,2.0,1.0,True,0.1,0.4,False,True
3,703548,2024-02-23,703548_2024-02-23_84353,3.0,1.0,False,0.1,0.4,False,False
4,703548,2024-02-23,703548_2024-02-23_84353,4.0,1.0,False,0.1,0.4,False,False
5,703548,2024-02-23,703548_2024-02-23_84353,5.0,1.0,False,0.1,0.4,False,False
6,703548,2024-02-23,703548_2024-02-23_84353,6.0,0.0,True,0.1,0.4,True,False
7,703548,2024-02-23,703548_2024-02-23_84353,7.0,0.0,False,0.1,0.4,False,False
8,703548,2024-02-23,703548_2024-02-23_84353,8.0,0.0,False,0.1,0.4,False,False
9,703548,2024-02-23,703548_2024-02-23_84353,9.0,1.0,True,0.1,0.4,False,True


## 5. Same pattern for events

In [16]:
df_events = duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, task, foraging_eff
        FROM read_parquet('{SESSION_DB}')
        WHERE task LIKE '%Uncoupled%'
          AND foraging_eff > 0.8
    )
    SELECT
        s.subject_id,
        s.session_date,
        s.task,
        e.session_id,
        e.timestamps,
        e.event,
        e.data
    FROM {READ_EVENTS} e
    JOIN sel s ON e.session_id = s._session_id
    WHERE CAST(e.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    ORDER BY s.subject_id, s.session_date, e.timestamps
""").df()

print(f"{len(df_events):,} events from {df_events['session_id'].nunique():,} sessions")
print(f"Event types: {sorted(df_events['event'].unique())}")
df_events.head(10)

13,402,631 events from 2,812 sessions
Event types: ['goCue_start_time', 'left_lick_time', 'left_reward_delivery_time', 'optogenetics_time', 'right_lick_time', 'right_reward_delivery_time']


,subject_id,session_date,task,session_id,timestamps,event,data
0,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,6.2210,left_lick_time,1.0
1,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,13.0659,left_lick_time,1.0
2,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,13.2830,left_lick_time,1.0
3,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,33.7682,left_lick_time,1.0
4,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,33.9585,left_lick_time,1.0
5,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,38.3667,left_lick_time,1.0
6,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,38.5555,left_lick_time,1.0
7,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,57.7755,left_lick_time,1.0
8,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,57.7925,left_reward_delivery_time,1.0
9,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,58.1000,left_lick_time,1.0


## 6. Aggregate across sessions — all in SQL

In [17]:
duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, task, foraging_eff
        FROM read_parquet('{SESSION_DB}')
        WHERE task LIKE '%Uncoupled%'
          AND foraging_eff > 0.8
    )
    SELECT
        s.subject_id,
        COUNT(DISTINCT s._session_id)  AS n_sessions,
        COUNT(*)                       AS n_trials,
        AVG(t.earned_reward::DOUBLE)   AS mean_reward_rate,
        AVG(s.foraging_eff)            AS mean_foraging_eff
    FROM {READ_TRIALS} t
    JOIN sel s ON t.session_id = s._session_id
    WHERE CAST(t.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    GROUP BY s.subject_id
    ORDER BY mean_foraging_eff DESC
""").df()

,subject_id,n_sessions,n_trials,mean_reward_rate,mean_foraging_eff
0,766556,1,520,0.411538,1.052861
1,816995,4,44,0.659091,1.019733
2,707524,1,188,0.377660,1.002825
3,827183,3,32,0.781250,0.995615
4,812300,1,504,0.422619,0.985512
...,...,...,...,...,...
549,756566,2,1082,0.506470,0.802106
550,648585,2,1612,0.571340,0.800938
551,796263,1,604,0.503311,0.800842
552,781477,1,460,0.463043,0.800763


## 7. DuckDB basics (reference — dip in anytime)

DuckDB is an in-process SQL engine that queries parquet (local or `s3://`) directly — no server and no separate load step. Essentials:
- `duckdb.sql(query)` returns a **lazy relation**; nothing runs until you materialise it with `.df()` (→ pandas), `.fetchall()`, or by displaying it.
- Read parquet with `read_parquet('…/**/*.parquet', hive_partitioning=true, union_by_name=true)` — use the `READ_TRIALS` / `READ_EVENTS` snippets defined above.
- Only the columns and rows you ask for are read from S3 (projection + filter *pushdown*), so **select the columns you need** instead of `SELECT *`.

In [8]:
# List every column (name + type) WITHOUT reading any rows: DESCRIBE
duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{SESSION_DB}')").df()

,column_name,column_type,null,key,default,extra
0,subject_id,VARCHAR,YES,None,None,None
1,session_date,VARCHAR,YES,None,None,None
2,nwb_suffix,BIGINT,YES,None,None,None
3,session,DOUBLE,YES,None,None,None
4,rig,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
154,curriculum_version_group,VARCHAR,YES,None,None,None
155,_session_id,VARCHAR,YES,None,None,None
156,co_asset_id,VARCHAR,YES,None,None,None
157,co_s3_nwb_uri,VARCHAR,YES,None,None,None


In [9]:
# The trial table has ~100 columns (the union of all NWB readers). Count and list them.
trial_cols = duckdb.sql(f"DESCRIBE SELECT * FROM {READ_TRIALS}").df()
print(f"{len(trial_cols)} columns in the trial table")
trial_cols[["column_name", "column_type"]].head(25)

103 columns in the trial table


,column_name,column_type
0,ITI_beta,DOUBLE
1,ITI_duration,DOUBLE
2,ITI_max,DOUBLE
3,ITI_min,DOUBLE
4,animal_response,DOUBLE
5,auto_train_curriculum_name,VARCHAR
6,auto_train_curriculum_schema_version,VARCHAR
7,auto_train_curriculum_version,VARCHAR
8,auto_train_engaged,VARCHAR
9,auto_train_stage,VARCHAR


In [10]:
# Column names as a plain Python list (schema only, no scan) — handy when building queries
duckdb.sql(f"SELECT * FROM {READ_EVENTS}").columns

['timestamps',
 'data',
 'event',
 'subject_id',
 'session_date',
 'nwb_suffix',
 'session_id',
 'nwb_data_source',
 'raw_timestamps',
 'trial']

In [11]:
# Peek at a few rows; LIMIT avoids scanning the whole table
duckdb.sql(f"SELECT * FROM read_parquet('{SESSION_DB}') LIMIT 3").df()

,subject_id,session_date,nwb_suffix,session,rig,trainer,PI,curriculum_name,curriculum_version,current_stage_actual,...,current_stage_suggested,session_at_current_stage,decision,next_stage_suggested,if_overriden_by_trainer,curriculum_version_group,_session_id,co_asset_id,co_s3_nwb_uri,nwb_data_source
0,854145,2026-06-03,131806,7.0,447-3-D,madeline.tom,Kenta Hagihara,Uncoupled Baiting,2.3,STAGE_FINAL,...,STAGE_FINAL,3,STAY,STAGE_FINAL,False,v3,854145_2026-06-03_131806,627a5cbe-008d-431e-8379-93c0a785f092,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/6...,co_asset
1,844917,2026-06-03,125658,21.0,322_EPHYS4,margaret.lee,None,Uncoupled Baiting,2.3,GRADUATED,...,GRADUATED,5,STAY,GRADUATED,False,v3,844917_2026-06-03_125658,None,None,bonsai_s3
2,844634,2026-06-03,122723,20.0,446-8-C,dean.rette,Anna Lakunina,Uncoupled Baiting,2.3,STAGE_FINAL,...,STAGE_FINAL,2,ROLLBACK,STAGE_3,False,v3,844634_2026-06-03_122723,8079c508-e322-4a80-9625-bbd50cb73533,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/8...,co_asset


In [12]:
# SUMMARIZE = one-shot column stats (min / max / avg / std / approx-unique / null %, percentiles)
duckdb.sql(f"""
    SUMMARIZE SELECT animal_response, earned_reward, reward_probabilityL, reward_probabilityR
    FROM {READ_TRIALS}
""").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,animal_response,DOUBLE,0.0,2.0,3,0.6530747100315605,0.65014062859614,0.0,1.0,1.0,12541020,0.0
1,earned_reward,BOOLEAN,false,true,2,None,None,None,None,None,12541020,0.0
2,reward_probabilityL,DOUBLE,0.0,1.0,20517,0.38767489584456155,0.27488868995289023,0.1,0.4,0.7,12541020,0.0
3,reward_probabilityR,DOUBLE,0.0,1.0,19531,0.37642856505170585,0.2752717768098857,0.1,0.4,0.7,12541020,0.0


In [13]:
# DuckDB can query an existing pandas DataFrame by its variable name (no copy) —
# so you can mix SQL and pandas freely.
recent = duckdb.sql(f"SELECT * FROM read_parquet('{SESSION_DB}') LIMIT 500").df()
duckdb.sql("SELECT subject_id, session_date, task, foraging_eff "
           "FROM recent ORDER BY foraging_eff DESC LIMIT 5").df()

,subject_id,session_date,task,foraging_eff
0,844634,2026-05-28,Uncoupled Baiting,0.907761
1,823165,2026-04-29,Uncoupled Baiting,0.897551
2,844634,2026-05-12,Uncoupled Baiting,0.892588
3,846558,2026-06-03,Uncoupled Baiting,0.883211
4,823166,2026-04-30,Uncoupled Baiting,0.881558
